> **출처 고지**: 도로망·통행시간과 그 파생값(표·그림)은 © OpenStreetMap contributors, [ODbL 1.0](https://opendatacommons.org/licenses/odbl/1-0/)이다. 인구이동은 KOSIS 국내인구이동통계 2025(공공누리 제1유형, 출처표시), 통근·통학은 KOSIS 인구총조사 2020(DT_1PA2021, 20% 표본)이다. 그 밖의 자료 출처와 라이선스는 저장소의 `NOTICE.md`를 본다.

# 04_diagnostics — 0S 사후 탐색 진단 (A–F)

**탐색적(사전 판정과 무관, ADR 0013 7항).** 0S 판정(곡률 조정, 쌍곡 이점 없음, 중력모형 이상의 위계 없음)은 바뀌지 않는다. 이 노트북은 두 "없음"의 원인을 진단한다. `embed_lib.py`(고정된 판정 코드)는 불러 쓰기만 하고 수정하지 않으며, 새 계산은 `diag_lib.py`에 있다. 결과는 `diag_results.json`, 그림과 표는 `diag_outputs/`에 둔다. 무거운 산출물(임베딩 좌표, 재시작 캐시)은 `C:\dev\kfm-data\spike\diag\`에 둔다.

| 절 | 내용 |
|---|---|
| A | 밀도를 맞춘 대조군(통근·통학 247개): 같은 지지 집합 대조군(A1), 엣지 수를 맞춘 재표본 대조군(A2), 대조군 밀도 곡선(보조) |
| B | 밀도별 δ̄ 곡선: 양방향 쌍 중 ℓ_sym이 작은 상위 p만 남기고 폐포 |
| C | 차원 D = 5, 10의 쌍곡·유클리드 왜곡 비(2단계 최적화 규약 그대로, R_max 5) |
| D | 인구이동 OD(KOSIS 국내인구이동 2025, 229개)로 만든 d_flow: δ̄, 중력 대조군, 2·5차원 임베딩 |
| E | 반경별 지역 수 증가(공 성장): 다항식형과 지수형 R², 유효 차원 |
| F | 가설별 지지 여부(규칙은 코드에 적었고 사전 고정이 아니다) |

인구이동 원본은 사전 검증 세션이 받은 KOSIS 파일을 `C:\dev\kfm-data\raw\kosis_mig2025_20261018\`에 복사하고 `manifest.csv`에 적었다(MD5 확인).

In [1]:
# 04_diagnostics 설정 셀: 경로, 설정값, 환경, 보조 함수
import os

for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_name] = "1"  # numpy, torch를 불러오기 전에 고정한다(자식 프로세스가 물려받는다)
os.environ["PYTORCH_JIT"] = "0"  # 03_embedding과 같은 이유: 재시작 결과가 시드와 입력으로만 정해지게 한다

import datetime
import json
import math
import platform
import subprocess
import sys
import time
import warnings
from pathlib import Path

import geoopt
import numpy as np
import pandas as pd
import scipy
import torch
from IPython.display import display

import embed_lib as el
import diag_lib as dl

assert not torch.jit._state._enabled.enabled, "torch JIT가 켜져 있다"
SMOKE = os.environ.get("KFM_DIAG_SMOKE") == "1"  # 배선 점검 모드(반복 수를 줄이고 산출물을 임시 폴더에 둔다). 실제 실행에서는 False
N_WORKERS = int(os.environ.get("KFM_DIAG_WORKERS", "10"))  # 재시작 병렬 프로세스 수(각 1스레드). 결과는 시드로 결정되므로 이 값에 의존하지 않는다
M1, M2 = 0.5, 0.9  # S4 고정값(줄인 유클리드 해의 최대 노름)
CFG = {**el.CFG, "m1": M1, "m2": M2}  # 0S와 같은 설정. 차원(dim)만 C·D절에서 바꾼다
el.check_cfg(CFG)
CFG_RUN = {**CFG, "adam_max_iter": 150, "fin_maxiter": 150, "fin_maxfun": 300} if SMOKE else CFG

T_START = time.time()
NB_DIR = Path.cwd()  # notebooks/spike (nbconvert는 노트북 폴더에서 실행한다)
assert (NB_DIR / "embed_lib.py").exists() and (NB_DIR / "diag_lib.py").exists(), "notebooks/spike 폴더에서 실행해야 한다"
assert "kfm-spike" in sys.executable, "커널이 kfm-spike 환경이 아니다"
RAW = Path(r"C:\dev\kfm-data\raw")
S4 = Path(r"C:\dev\kfm-data\spike\s4")
S4_REAL = S4 / "s4_real"
ORIGINS = Path(r"C:\dev\kfm-data\spike\origins.parquet")
if SMOKE:
    SCRATCH = Path(os.environ["KFM_DIAG_SCRATCH"])
    DIAG, OUT_FIG, RESULT_JSON = SCRATCH / "data", SCRATCH / "figs", SCRATCH / "diag_results.json"
else:
    DIAG, OUT_FIG, RESULT_JSON = Path(r"C:\dev\kfm-data\spike\diag"), NB_DIR / "diag_outputs", NB_DIR / "diag_results.json"
for _p in (DIAG, OUT_FIG):
    _p.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 400)

R = {"label": dl.EXPLORATORY, "note": "0S 판정은 바뀌지 않는다. 원인 진단용 결과이며 판정 규칙의 기준값이 아니다.", "meta": {}}
if not SMOKE and RESULT_JSON.exists():
    try:  # 같은 노트북의 이전 실행 구간별 시간을 남긴다. 재시작은 캐시(restart_cache)에서 읽으므로, 재시작을 새로 계산한 실행(임베딩 구간 10분 초과)의 시간은 compute_run_seconds로 따로 이어 둔다
        prev_ = json.loads(RESULT_JSON.read_text(encoding="utf-8"))
        R["meta"]["previous_run_seconds"] = prev_.get("seconds")
        cands_ = [prev_.get("meta", {}).get("compute_run_seconds"), prev_.get("seconds"), prev_.get("meta", {}).get("previous_run_seconds")]
        R["meta"]["compute_run_seconds"] = next((c_ for c_ in cands_ if c_ and c_.get("C·D2 임베딩", 0) > 600), None)
    except (OSError, ValueError, AttributeError):
        pass
SECONDS = {}
_lap = {"t": time.time()}


def lap(name):
    now = time.time()
    SECONDS[name] = round(now - _lap["t"], 1)
    _lap["t"] = now
    print(f"[{name}] {SECONDS[name]:.1f} s (누적 {now - T_START:.0f} s)")


def save_results():
    R["seconds"] = dict(SECONDS)
    RESULT_JSON.write_text(json.dumps(dl.json_clean(R), ensure_ascii=False, indent=1), encoding="utf-8")


def kd(d):
    # {(정규화, 부분집합): 값} -> {"정규화|부분집합": 값}
    return {dl.cond_key(*c): v for c, v in d.items()}


def stats_kd(d):
    # {(정규화, 부분집합): 배열} -> {"정규화|부분집합": 요약}
    return {dl.cond_key(*c): dl.stats_of(v) for c, v in d.items()}


def csv_out(df, name):
    df.to_csv(OUT_FIG / name, index=False, encoding="utf-8-sig")


print("실행 시각:", datetime.datetime.now().isoformat(timespec="seconds"), "| python", sys.version.split()[0], "|", platform.platform())
print("실행 파일:", sys.executable, "| SMOKE =", SMOKE, "| N_WORKERS =", N_WORKERS)
print("torch", torch.__version__, "| geoopt", geoopt.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__, "| pandas", pd.__version__, "| numba", el.numba.__version__)
print("설정:", CFG)
print("config_hash:", el.config_hash(CFG))
print("라벨:", dl.EXPLORATORY)

C:\dev\kfm-spike\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


실행 시각: 2026-10-19T23:50:48 | python 3.11.9 | Windows-10-10.0.26200-SP0
실행 파일: C:\dev\kfm-spike\.venv\Scripts\python.exe | SMOKE = False | N_WORKERS = 10
torch 2.14.1+cpu | geoopt 0.5.1 | numpy 2.4.6 | scipy 1.17.1 | pandas 3.0.6 | numba 0.68.0
설정: {'dim': 2, 'init_halfwidth': 0.1, 'adam_lr': 0.05, 'adam_window': 200, 'adam_tol': 1e-06, 'adam_max_iter': 50000, 'r_max': 5.0, 'fin_ftol': 1e-12, 'fin_gtol': 1e-10, 'fin_maxiter': 20000, 'fin_maxfun': 50000, 'mds_max_norm': 0.14142135623730953, 'm1': 0.5, 'm2': 0.9, 'n_euc_u': 4, 'n_hyp_u': 3, 'retry_offsets': (5, 10), 'flat_rtol': 0.001, 'ratio_max': 0.8, 'n_worst': 20, 'tie_rtol': 1e-12, 'resample_frac': 0.2, 'resample_scale': 5}
config_hash: c2ae3329a5afc3d944bb26ba125c1ad4
라벨: 탐색적(사전 판정과 무관, ADR 0013 7항)


## 1절. 입력 (S4가 고정한 행렬, 인구이동 원본)

S4가 고정한 행렬(`C:\dev\kfm-data\spike\s4\`)은 `s4_freeze.json`의 MD5와 같아야 한다. 인구이동 원본은 `manifest.csv`의 MD5와 같아야 한다.

In [2]:
FREEZE = json.loads((NB_DIR / "s4_freeze.json").read_text(encoding="utf-8"))
inp = dl.load_s4_inputs(S4, NB_DIR / "s4_freeze.json")  # MD5가 다르면 여기서 멈춘다
ISL, T247, T_EXP = inp["island247"], inp["od_247"], inp["gravity_expected_247"]
manifest = pd.read_csv(RAW / "manifest.csv", encoding="utf-8-sig", dtype=str).set_index("path")
MIG_REL = "kosis_mig2025_20261018/mig2025.xlsx"
MIG_XLSX = RAW / MIG_REL
MIG_MD5 = dl.md5_file(MIG_XLSX)
assert MIG_MD5 == manifest.loc[MIG_REL, "md5"], "인구이동 원본이 manifest와 다르다"
R["meta"]["mean_time_obs_min"] = FREEZE["gravity"]["mean_time_obs_min"]
R["meta"]["beta_per_min_commute"] = FREEZE["gravity"]["beta_per_min"]
print("S4 입력", len(inp["md5"]), "개 MD5 일치 | 인구이동 원본 MD5 일치:", MIG_MD5)
print(f"통근·통학 247개: 관측 양방향 엣지 {dl.n_edges(T247):,}개, 존 간 합 {el.od_margins(T247)[0].sum():,.0f}명, 섬 연결 지역 {int(ISL.sum())}개, 중력 β = {FREEZE['gravity']['beta_per_min']:.6f} /분")
lap("1 입력")

S4 입력 13 개 MD5 일치 | 인구이동 원본 MD5 일치: 9f60e162d88b87d7a4dbaa8f0ca1681c
통근·통학 247개: 관측 양방향 엣지 6,815개, 존 간 합 9,501,939명, 섬 연결 지역 1개, 중력 β = 0.100157 /분
[1 입력] 0.1 s (누적 0 s)


## A. 밀도를 맞춘 대조군 (통근·통학 247개) — 탐색적

0S 대조군은 20% 재표본 뒤 관측 0 패턴을 적용해 양방향 엣지가 관측보다 약 23% 적었다. 성긴 그래프는 폐포 뒤 트리에 가까워 δ̄가 낮게 나오기 쉬우므로, 대조군 밀도를 관측에 맞춰 위계 판정이 바뀌는지 본다.

- **A1 같은 지지 집합**: 중력모형 기댓값 T̂를 재표본 없이 관측의 양방향 엣지 집합에만 두고 나머지는 0으로 한다.
- **A2 엣지 수를 맞춘 재표본**: 유효 표본 배율 k(0S의 표본 20%에 곱함)를 이분법으로 정해, 관측 0 패턴을 적용한 재표본 대조군의 평균 양방향 엣지 수가 관측(6,815)의 ±1% 안에 들게 한다. 엣지 수는 관측 지지 집합 크기에 점근적으로만 접근하므로 허용 대역의 가운데(−0.5%)를 목표로 한다. 20개를 만든다.
- **A3(보조)**: 표본 배율 k를 키워 가며 대조군 밀도에 따른 δ̄를 직접 본다.

In [3]:
# A-1. 관측과 0S 대조군의 δ̄를 다시 계산해 S4 저장값과 맞는지 확인한다
OBS = dl.delta_bars(inp["d_flow_main_247"], inp["d_flow_alt_247"], ISL, CFG)
CTRL0 = el.control_delta_bars({"main": list(inp["controls_d_main_247"]), "alt": list(inp["controls_d_alt_247"])}, ISL, CFG)
saved_ = np.load(S4_REAL / "control_delta_bars.npz")
assert all(np.allclose(CTRL0[c], saved_[dl.cond_key(*c)], atol=1e-12, rtol=0) for c in dl.COND), "0S 대조군 δ̄가 S4 저장값과 다르다"
OBS_EDGES = dl.n_edges(T247)
CTRL0_EDGES = json.loads((S4 / "controls_meta.json").read_text(encoding="utf-8"))["edges"]
print("관측 δ̄:", {k: round(v, 4) for k, v in kd(OBS).items()}, "| 엣지", OBS_EDGES)
print("0S 대조군 20개 평균 엣지", np.mean(CTRL0_EDGES), "(관측 대비", round(np.mean(CTRL0_EDGES) / OBS_EDGES, 4), ") — S4 저장값과 일치")

# A-2. A1: 같은 지지 집합 대조군
T_A1 = dl.same_support_control(T_EXP, T247)
FD_A1 = el.flow_distance(T_A1, kind="estimated", role=None)
A1 = dl.delta_bars(FD_A1["d_main"], FD_A1["d_alt"], ISL, CFG)
A1_EDGES = dl.n_edges(T_A1)
assert A1_EDGES == OBS_EDGES
print("A1 δ̄:", {k: round(v, 4) for k, v in kd(A1).items()}, "| 엣지", A1_EDGES)

# A-3. A2: 표본 배율을 이분법으로 정해 엣지 수를 맞춘 재표본 대조군 20개
MATCH = dl.match_sample_multiplier(T_EXP, T247, CFG)
assert MATCH["ok"], MATCH
CTRL_A2, SKIP_A2 = dl.make_matched_controls(T_EXP, T247, MATCH["frac"], CFG)
A2 = el.control_delta_bars({"main": [c["d_main"] for c in CTRL_A2], "alt": [c["d_alt"] for c in CTRL_A2]}, ISL, CFG)
A2_EDGES = [c["n_edges"] for c in CTRL_A2]
assert abs(np.mean(A2_EDGES) / OBS_EDGES - 1) <= 0.01, "A2 대조군 평균 엣지 수가 관측 +-1%를 벗어났다"
print(f"A2: 표본 배율 k = {MATCH['k']:.2f} (표본 비율 {MATCH['frac']:.1f}, 유효 표본 수 {CTRL_A2[0]['n_eff']:,}), 평균 엣지 {np.mean(A2_EDGES):.1f} (관측 대비 {np.mean(A2_EDGES) / OBS_EDGES:.4f}), "
      f"허용 대역 {MATCH['band'][0]:.1f}–{MATCH['band'][1]:.1f}, 건너뛴 시드 {len(SKIP_A2)}개, 이분법 평가 {len(MATCH['history'])}회")
print("A2 δ̄:", {k: (round(v['mean'], 4), round(v['min'], 4), round(v['max'], 4)) for k, v in stats_kd(A2).items()})
lap("A-1~3 관측·0S·A1·A2")

관측 δ̄: {'main|전체': 0.3012, 'main|섬 제외': 0.3007, 'alt|전체': 0.3007, 'alt|섬 제외': 0.3003} | 엣지 6815
0S 대조군 20개 평균 엣지 5216.55 (관측 대비 0.7655 ) — S4 저장값과 일치


A1 δ̄: {'main|전체': 0.2597, 'main|섬 제외': 0.259, 'alt|전체': 0.2651, 'alt|섬 제외': 0.2644} | 엣지 6815


A2: 표본 배율 k = 2041.08 (표본 비율 408.2, 유효 표본 수 3,878,843,996), 평균 엣지 6781.1 (관측 대비 0.9950), 허용 대역 6746.9–6883.1, 건너뛴 시드 0개, 이분법 평가 22회
A2 δ̄: {'main|전체': (0.2594, 0.2592, 0.2596), 'main|섬 제외': (0.2588, 0.2586, 0.259), 'alt|전체': (0.2648, 0.2647, 0.2649), 'alt|섬 제외': (0.2642, 0.264, 0.2643)}
[A-1~3 관측·0S·A1·A2] 28.3 s (누적 28 s)


In [4]:
# A-4. 한 표로 비교, 위계 판정 점검, 대조군 밀도 곡선(A3), 그림
HIER_0S = el.judge_hierarchy(OBS, CTRL0, 20)
HIER_A1 = el.judge_hierarchy(OBS, {c: np.array([A1[c]]) for c in dl.COND}, 1)
HIER_A2 = el.judge_hierarchy(OBS, A2, 20)
A_TABLE = dl.comparison_table(
    [("관측", OBS_EDGES, OBS), ("기존 대조군(0S, 표본 20%)", np.mean(CTRL0_EDGES), CTRL0), ("A1 같은 지지 집합(재표본 없음)", A1_EDGES, A1),
     (f"A2 엣지 맞춤 재표본(표본 배율 x{MATCH['k']:.0f})", np.mean(A2_EDGES), A2)], OBS)
display(A_TABLE)
print("위계 판정(4조건 모두 관측 < 대조군 최솟값): 0S", HIER_0S["hierarchy"], "| A1", HIER_A1["hierarchy"], "| A2", HIER_A2["hierarchy"])
display(HIER_A2["table"].round(4))
SWEEP = dl.control_density_sweep(T_EXP, T247, [1, 4, 16, 64, 256, 1024, MATCH["k"]], range(6), ISL, CFG)
display(SWEEP[["k", "n_eff", "n_controls", "skipped", "mean_edges", "main|전체_mean", "main|전체_min", "main|전체_max", "alt|전체_mean"]].round(4))
A2_STATS = stats_kd(A2)
dl.plot_A(SWEEP, OBS, OBS_EDGES, A1, {"mean_edges": float(np.mean(A2_EDGES)), "k": MATCH["k"], "bars": A2_STATS}, OUT_FIG / "fig_A_density_controls.png")
csv_out(A_TABLE, "table_A_controls.csv")
csv_out(SWEEP, "table_A_sweep.csv")
TH_ = T_EXP[el.flow_edge_mask(T247)]  # 관측 양방향 엣지 셀(방향별)의 중력 기댓값 T̂: 작은 값이 많을수록 재표본 대조군이 그 엣지를 못 만든다
T_HAT_EDGES = {"n_cells": int(TH_.size), "min": float(TH_.min()), "q01": float(np.quantile(TH_, 0.01)), "median": float(np.median(TH_)),
               **{f"n_lt_{x_:g}": int((TH_ < x_).sum()) for x_ in (1e-6, 1e-3, 1e-2, 0.1, 1.0)}}
print("관측 엣지 셀의 중력 기댓값 T̂:", {k_: (round(v_, 8) if isinstance(v_, float) else v_) for k_, v_ in T_HAT_EDGES.items()})
R["A"] = {
    "t_hat_on_observed_edges": T_HAT_EDGES,
    "label": dl.EXPLORATORY,
    "obs_edges": OBS_EDGES,
    "observed": kd(OBS),
    "ctrl_0S": {**stats_kd(CTRL0), "mean_edges": float(np.mean(CTRL0_EDGES)), "edges": CTRL0_EDGES},
    "A1": {"edges": A1_EDGES, "bars": kd(A1)},
    "A2": {"k": MATCH["k"], "frac": MATCH["frac"], "n_eff": CTRL_A2[0]["n_eff"], "mean_edges": float(np.mean(A2_EDGES)), "edges": A2_EDGES,
           "band": list(MATCH["band"]), "goal": MATCH["goal"], "skipped_seeds": [s["seed"] for s in SKIP_A2], "n_bisect_evals": len(MATCH["history"]),
           "bars": A2_STATS, "values": {dl.cond_key(*c): A2[c].tolist() for c in dl.COND}},
    "hierarchy_0S": {"hierarchy": HIER_0S["hierarchy"], "table": HIER_0S["table"].to_dict("records")},
    "hierarchy_A1": {"hierarchy": HIER_A1["hierarchy"], "table": HIER_A1["table"].to_dict("records")},
    "hierarchy_A2": {"hierarchy": HIER_A2["hierarchy"], "table": HIER_A2["table"].to_dict("records")},
    "table": A_TABLE.to_dict("records"),
    "sweep": SWEEP.to_dict("records"),
}
save_results()
lap("A-4 비교표·밀도 곡선·그림")

,대상,엣지(평균),n,δ̄ 주-전체,δ̄ 주-섬 제외,δ̄ 대안-전체,δ̄ 대안-섬 제외,관측 이하 대조군 수(4조건)
0,관측,6815.0,1,0.3012,0.3007,0.3007,0.3003,-
1,"기존 대조군(0S, 표본 20%)",5216.6,20,0.2625 [0.2598-0.2644],0.2619 [0.2592-0.2638],0.2651 [0.2623-0.2668],0.2644 [0.2617-0.2661],20/20/20/20
2,A1 같은 지지 집합(재표본 없음),6815.0,1,0.2597,0.2590,0.2651,0.2644,1/1/1/1
3,A2 엣지 맞춤 재표본(표본 배율 x2041),6781.1,20,0.2594 [0.2592-0.2596],0.2588 [0.2586-0.2590],0.2648 [0.2647-0.2649],0.2642 [0.2640-0.2643],20/20/20/20


위계 판정(4조건 모두 관측 < 대조군 최솟값): 0S False | A1 False | A2 False


,정규화,부분집합,d_flow,대조군 최솟값,대조군 중앙값,대조군 최댓값,d_flow 이하인 대조군 수,여유(최솟값 - d_flow),통과
0,main,전체,0.3012,0.2592,0.2594,0.2596,20,-0.0419,False
1,main,섬 제외,0.3007,0.2586,0.2588,0.2590,20,-0.0421,False
2,alt,전체,0.3007,0.2647,0.2648,0.2649,20,-0.0360,False
3,alt,섬 제외,0.3003,0.2640,0.2642,0.2643,20,-0.0363,False


,k,n_eff,n_controls,skipped,mean_edges,main|전체_mean,main|전체_min,main|전체_max,alt|전체_mean
0,1.0000,1900388,6,0,5209.3333,0.2621,0.2598,0.2644,0.2645
1,4.0000,7601551,6,0,5925.6667,0.2603,0.2579,0.2622,0.2645
2,16.0000,30406205,6,0,6337.8333,0.2604,0.2598,0.2608,0.2648
3,64.0000,121624819,6,0,6585.5000,0.2595,0.2593,0.2598,0.2647
4,256.0000,486499277,6,0,6709.6667,0.2593,0.2591,0.2596,0.2647
5,1024.0000,1945997107,6,0,6764.0000,0.2595,0.2594,0.2597,0.2649
6,2041.0802,3878843996,6,0,6782.0000,0.2594,0.2592,0.2595,0.2648


관측 엣지 셀의 중력 기댓값 T̂: {'n_cells': 13630, 'min': 2e-07, 'q01': 0.01861796, 'median': 82.40121524, 'n_lt_1e-06': 3, 'n_lt_0.001': 31, 'n_lt_0.01': 103, 'n_lt_0.1': 346, 'n_lt_1': 1241}
[A-4 비교표·밀도 곡선·그림] 27.9 s (누적 56 s)


## B. 밀도별 δ̄ 곡선 — 탐색적

p = 0.1, 0.2, …, 1.0마다 양방향 쌍 중 ℓ_sym(폐포 전)이 작은(상대 흐름이 강한) 상위 p만 남기고 최단경로 폐포를 한 뒤 δ̄를 구한다. 관측과 A1 대조군에 각자의 ℓ 순위로 똑같이 한다. 연결이 끊기면 최대 성분만 쓰고 n을 적는다. p = 1.0은 A절의 전체 δ̄(주 정규화)와 같다.

In [5]:
FD_OBS = el.flow_distance(T247, kind="observed", role="observed")
assert np.allclose(FD_OBS["d_main"], inp["d_flow_main_247"], atol=1e-12)
CURVE_OBS = dl.density_curve(FD_OBS["raw_main"])
CURVE_A1 = dl.density_curve(FD_A1["raw_main"])
assert abs(CURVE_OBS["δ̄"].iloc[-1] - OBS[("main", "전체")]) < 1e-12 and abs(CURVE_A1["δ̄"].iloc[-1] - A1[("main", "전체")]) < 1e-12, "p = 1.0이 A절 값과 다르다"
cols_ = ["p", "남긴 쌍", "연결 성분", "고립 지역", "n(최대 성분)", "엣지(최대 성분)", "δ̄", "δ_rel", "동률 네 점"]
print("관측"); display(CURVE_OBS[cols_].round(4))
print("A1 대조군(같은 지지 집합)"); display(CURVE_A1[cols_].round(4))
GAP_B = (CURVE_OBS["δ̄"] - CURVE_A1["δ̄"]).round(4).tolist()
print("관측 - A1 δ̄ (p = 0.1 ... 1.0):", GAP_B)
dl.plot_B({"관측": CURVE_OBS, "A1 대조군(같은 지지 집합)": CURVE_A1}, OUT_FIG / "fig_B_density_curve.png")
csv_out(pd.concat([CURVE_OBS.assign(대상="관측"), CURVE_A1.assign(대상="A1")]), "table_B_density_curve.csv")
R["B"] = {"label": dl.EXPLORATORY, "observed": CURVE_OBS.to_dict("records"), "A1": CURVE_A1.to_dict("records"), "gap_obs_minus_A1": GAP_B,
          "rule": "양방향 쌍을 폐포 전 ℓ_sym 오름차순으로 세워 상위 round(p x 쌍 수)개만 남기고 최단경로 폐포. 연결이 끊기면 최대 성분만 쓰고 n을 적는다. 주 정규화"}
save_results()
lap("B 밀도별 δ̄ 곡선")

관측


,p,남긴 쌍,연결 성분,고립 지역,n(최대 성분),엣지(최대 성분),δ̄,δ_rel,동률 네 점
0,0.1,682,3,2,245,682,0.2623,0.6072,157656
1,0.2,1363,1,0,247,1363,0.2627,0.6387,71935
2,0.3,2045,1,0,247,2045,0.2635,0.6442,46608
3,0.4,2726,1,0,247,2726,0.2672,0.6210,40348
4,0.5,3408,1,0,247,3408,0.2659,0.6268,43883
5,0.6,4089,1,0,247,4089,0.2698,0.6339,34892
6,0.7,4771,1,0,247,4771,0.2737,0.6395,31870
7,0.8,5452,1,0,247,5452,0.2731,0.6583,31324
8,0.9,6134,1,0,247,6134,0.2728,0.6816,26696
9,1.0,6815,1,0,247,6815,0.3012,0.7223,18628


A1 대조군(같은 지지 집합)


,p,남긴 쌍,연결 성분,고립 지역,n(최대 성분),엣지(최대 성분),δ̄,δ_rel,동률 네 점
0,0.1,682,4,3,244,682,0.2618,0.5667,134332
1,0.2,1363,2,1,246,1363,0.2651,0.6200,46884
2,0.3,2045,1,0,247,2045,0.2631,0.6700,29126
3,0.4,2726,1,0,247,2726,0.2627,0.6503,24901
4,0.5,3408,1,0,247,3408,0.2611,0.6525,19587
5,0.6,4089,1,0,247,4089,0.2619,0.6445,18989
6,0.7,4771,1,0,247,4771,0.2610,0.6679,16535
7,0.8,5452,1,0,247,5452,0.2597,0.6421,15778
8,0.9,6134,1,0,247,6134,0.2592,0.6617,14721
9,1.0,6815,1,0,247,6815,0.2597,0.6677,14697


관측 - A1 δ̄ (p = 0.1 ... 1.0): [0.0006, -0.0024, 0.0005, 0.0046, 0.0048, 0.0079, 0.0127, 0.0133, 0.0137, 0.0415]
[B 밀도별 δ̄ 곡선] 3.5 s (누적 60 s)


## D1. 인구이동 OD(KOSIS 국내인구이동 2025) 준비, δ̄, 중력 대조군 — 탐색적

- 단위는 229개(일반구를 상위 시에 합산). `d_time` 229개는 250개 행렬의 하위 지역 쌍 인구 가중 평균(`merge_weighted`)이다. 전국 이동자 6,117,784명(PLAN 2-6b 검증값, 존 내 이동 포함)을 확인한다.
- `d_flow` 주·대안 정규화는 존 내 이동을 뺀 OD로 만든다. δ, δ_rel, δ̄는 전체와 섬 연결 지역(제주시·서귀포시·울릉군) 제외에서 구한다.
- 중력 대조군: β는 229개 `d_time`의 이동자 가중 평균 통행시간(방향 통행시간)에 맞춘다. 이동 통계는 전수 자료이므로 ADR 0010 6항의 표본 수 경로(같은 총량 재표본 = 포아송의 총량 조건부 분포(다항), 관측 0 패턴)로 20개를 만들고, A1과 같은 같은 지지 집합 대조군 1개도 만든다. 임베딩은 D2절에서 한꺼번에 돌린다.

In [6]:
MIG = dl.prepare_migration(inp, MIG_XLSX)
T_MIG, ISL_MIG = MIG["T"], MIG["island"]
print("전국 이동자 합(존 내 포함):", f"{MIG['checks']['sum_sgg_cells']:,.0f}", "= 공표값 6,117,784 | 존 내 이동", f"{MIG['checks']['within_sgg_diag_sum']:,.0f}", "| 존 간", f"{MIG['checks']['between_sgg_sum']:,.0f}")
print("합친 시:", MIG["merged_units"], "개(하위 지역", MIG["merged_subunits"], "개) | 섬 연결 지역:", [MIG["labels229"][i] for i in np.flatnonzero(ISL_MIG)])
FX_MIG = el.flow_exclusion(T_MIG, ISL_MIG)
assert FX_MIG["keep"].all(), "OD에서 빠진 지역이 있다"
T_MIG0 = el.od_margins(T_MIG)[0]
print(f"존 간 셀 {229 * 228:,}개 중 0인 셀 {int(((T_MIG0 == 0) & ~np.eye(229, dtype=bool)).sum()):,}개, 양방향 엣지 {dl.n_edges(T_MIG):,}개 / 쌍 {229 * 228 // 2:,}개")
FD_MIG = el.flow_distance(T_MIG, kind="observed", role="observed")
DELTA_MIG = el.delta_table({"d_time_229": {"D": MIG["d_time"], "island": ISL_MIG}, "d_flow_main_mig": {"D": FD_MIG["d_main"], "island": ISL_MIG},
                            "d_flow_alt_mig": {"D": FD_MIG["d_alt"], "island": ISL_MIG}}, CFG)
display(DELTA_MIG[["입력", "부분집합", "n", "diam", "delta", "delta_rel", "delta_bar", "n_ties", "delta_diff"]].round(4))
OBS_MIG = {(norm, sub): float(DELTA_MIG[(DELTA_MIG["입력"] == nm) & (DELTA_MIG["부분집합"] == sub)]["delta_bar"].iloc[0])
           for norm, nm in (("main", "d_flow_main_mig"), ("alt", "d_flow_alt_mig")) for sub in dl.SUBSETS}
FIT_MIG = el.fit_gravity(T_MIG, MIG["t_dir"])
print(f"중력모형 적합: β = {FIT_MIG['beta']:.8f} /분, 평균 방향 통행시간 관측 {FIT_MIG['mean_time_obs']:.4f} 분 = 모형 {FIT_MIG['mean_time_fit']:.4f} 분, Furness {FIT_MIG['furness_iter']}회")
CFG_MIG = {**CFG, "resample_frac": 1.0, "resample_scale": 1.0}  # ADR 0010 6항 표본 수 경로: 같은 총량(존 간 합)으로 재표본, 관측 0 패턴 적용
CTRL_MIG, SKIP_MIG = el.make_controls(FIT_MIG["T_exp"], T_MIG, n_wanted=20, cfg=CFG_MIG)
assert CTRL_MIG[0]["n_eff"] == int(math.floor(T_MIG0.sum() + 0.5))
CTRL_MIG_BARS = el.control_delta_bars({"main": [c["d_main"] for c in CTRL_MIG], "alt": [c["d_alt"] for c in CTRL_MIG]}, ISL_MIG, CFG)
CTRL_MIG_EDGES = [c["n_edges"] for c in CTRL_MIG]
T_A1_MIG = dl.same_support_control(FIT_MIG["T_exp"], T_MIG)
FD_A1_MIG = el.flow_distance(T_A1_MIG, kind="estimated", role=None)
A1_MIG = dl.delta_bars(FD_A1_MIG["d_main"], FD_A1_MIG["d_alt"], ISL_MIG, CFG)
HIER_MIG = el.judge_hierarchy(OBS_MIG, CTRL_MIG_BARS, 20)
MIG_TABLE = dl.comparison_table(
    [("관측(인구이동)", dl.n_edges(T_MIG), OBS_MIG), ("중력 대조군(같은 총량 재표본, 20개)", np.mean(CTRL_MIG_EDGES), CTRL_MIG_BARS),
     ("A1형 같은 지지 집합(재표본 없음)", dl.n_edges(T_A1_MIG), A1_MIG)], OBS_MIG)
display(MIG_TABLE)
display(HIER_MIG["table"].round(4))
print("중력모형 이상의 위계(인구이동, 4조건 모두):", HIER_MIG["hierarchy"], "| 건너뛴 시드", [s["seed"] for s in SKIP_MIG])
D_TIME_229_DB = float(DELTA_MIG[(DELTA_MIG["입력"] == "d_time_229") & (DELTA_MIG["부분집합"] == "전체")]["delta_bar"].iloc[0])
csv_out(DELTA_MIG.drop(columns=["seconds"]), "table_D_migration_delta.csv")
csv_out(MIG_TABLE, "table_D_migration_controls.csv")
R["D"] = {
    "label": dl.EXPLORATORY,
    "source": {"file": MIG_REL, "md5": MIG_MD5, "note": "KOSIS 국내인구이동통계 2025년(연간) 시군구 전출입자수(계)"},
    "checks": MIG["checks"], "n_units": 229, "merged_units": MIG["merged_units"], "merged_subunits": MIG["merged_subunits"],
    "island_units": [MIG["labels229"][i] for i in np.flatnonzero(ISL_MIG)],
    "edges": dl.n_edges(T_MIG), "zero_offdiag_cells": int(((T_MIG0 == 0) & ~np.eye(229, dtype=bool)).sum()),
    "gravity": {"beta_per_min": FIT_MIG["beta"], "mean_time_obs_min": FIT_MIG["mean_time_obs"], "mean_time_fit_min": FIT_MIG["mean_time_fit"], "furness_iter": FIT_MIG["furness_iter"],
                "resample": "같은 총량(존 간 합) 다항 재표본(포아송의 총량 조건부) + 관측 0 패턴, 시드 0.. (embed_lib.make_controls, frac 1.0, scale 1)",
                "n_eff": CTRL_MIG[0]["n_eff"], "seeds": [c["seed"] for c in CTRL_MIG], "skipped_seeds": [s["seed"] for s in SKIP_MIG], "control_edges": CTRL_MIG_EDGES},
    "delta_table": DELTA_MIG.drop(columns=["seconds"]).to_dict("records"),
    "observed": kd(OBS_MIG), "controls": stats_kd(CTRL_MIG_BARS), "control_values": {dl.cond_key(*c): CTRL_MIG_BARS[c].tolist() for c in dl.COND},
    "A1": {"edges": dl.n_edges(T_A1_MIG), "bars": kd(A1_MIG)},
    "hierarchy": {"hierarchy": HIER_MIG["hierarchy"], "table": HIER_MIG["table"].to_dict("records")},
    "d_time_229_delta_bar": D_TIME_229_DB,
    "table": MIG_TABLE.to_dict("records"),
}
save_results()
lap("D1 인구이동 준비·δ̄·대조군")

전국 이동자 합(존 내 포함): 6,117,784 = 공표값 6,117,784 | 존 내 이동 2,095,832 | 존 간 4,021,952
합친 시: 11 개(하위 지역 32 개) | 섬 연결 지역: ['경북 울릉군', '제주 제주시', '제주 서귀포시']
존 간 셀 52,212개 중 0인 셀 4,201개, 양방향 엣지 22,896개 / 쌍 26,106개


,입력,부분집합,n,diam,delta,delta_rel,delta_bar,n_ties,delta_diff
0,d_time_229,전체,229,621.4297,146.9738,0.4730,0.2489,0,0.0
1,d_time_229,섬 제외,226,428.6152,100.3104,0.4681,0.2482,0,0.0
2,d_flow_main_mig,전체,229,11.8796,4.5147,0.7601,0.3835,769,0.0
3,d_flow_main_mig,섬 제외,226,11.8796,4.5147,0.7601,0.3823,751,0.0
4,d_flow_alt_mig,전체,229,11.8579,4.4765,0.7550,0.3843,929,0.0
5,d_flow_alt_mig,섬 제외,226,11.8579,4.4765,0.7550,0.3831,911,0.0


중력모형 적합: β = 0.01977279 /분, 평균 방향 통행시간 관측 60.9574 분 = 모형 60.9574 분, Furness 75회


,대상,엣지(평균),n,δ̄ 주-전체,δ̄ 주-섬 제외,δ̄ 대안-전체,δ̄ 대안-섬 제외,관측 이하 대조군 수(4조건)
0,관측(인구이동),22896.0,1,0.3835,0.3823,0.3843,0.3831,-
1,"중력 대조군(같은 총량 재표본, 20개)",21372.8,20,0.3706 [0.3699-0.3716],0.3699 [0.3693-0.3708],0.3714 [0.3708-0.3723],0.3706 [0.3701-0.3715],20/20/20/20
2,A1형 같은 지지 집합(재표본 없음),22896.0,1,0.3340,0.3341,0.3346,0.3346,1/1/1/1


,정규화,부분집합,d_flow,대조군 최솟값,대조군 중앙값,대조군 최댓값,d_flow 이하인 대조군 수,여유(최솟값 - d_flow),통과
0,main,전체,0.3835,0.3699,0.3706,0.3716,20,-0.0136,False
1,main,섬 제외,0.3823,0.3693,0.3698,0.3708,20,-0.0131,False
2,alt,전체,0.3843,0.3708,0.3713,0.3723,20,-0.0135,False
3,alt,섬 제외,0.3831,0.3701,0.3705,0.3715,20,-0.0130,False


중력모형 이상의 위계(인구이동, 4조건 모두): False | 건너뛴 시드 []
[D1 인구이동 준비·δ̄·대조군] 33.1 s (누적 93 s)


## C·D2. 임베딩 일괄 실행 — 탐색적

2단계 최적화 규약(ADR 0013 4항) 그대로다: R_max 5, 유클리드 5회(고전 MDS 1 + U 시드 0..3) + 쌍곡 5회(줄인 초기값 2 + U 시드 0..2), RiemannianAdam 뒤 L-BFGS-B 마무리, 실패 시 쌍곡 U 재시작만 시드 +5, +10으로 최대 2회 재실행. 차원 D만 `cfg["dim"]`으로 바꾼다(`embed_lib`의 수치 함수를 그대로 쓴다).

- C: `d_time` 247개와 `d_flow` 주 정규화(통근·통학) 247개를 D = 5, 10으로, 0S 중력 대조군 2개(`ctrl_main_0`, `ctrl_main_1`)를 D = 5로. D = 2는 0S 결과를 쓴다(재계산 없음).
- D: 인구이동 `d_flow` 주 정규화 229개와 중력 대조군 2개를 D = 2, 5로.

12개 입력을 한 번에 프로세스 풀에 보낸다. 재시작 결과는 `C:\dev\kfm-data\spike\diag\restart_cache\`에 쌓이므로 중간에 멈춰도 이어서 돌릴 수 있다.

In [7]:
PROB = {}
for base_, M_ in (("d_time_247", inp["d_time_247"]), ("d_flow_main", inp["d_flow_main_247"])):
    for dim_ in (5, 10):
        PROB[f"{base_}_D{dim_}"] = (M_, dim_)
for r_ in (0, 1):
    PROB[f"ctrl_main_{r_}_D5"] = (inp["controls_d_main_247"][r_], 5)
for dim_ in (2, 5):
    PROB[f"mig_obs_D{dim_}"] = (FD_MIG["d_main"], dim_)
    for r_ in (0, 1):
        PROB[f"mig_ctrl_main_{r_}_D{dim_}"] = (CTRL_MIG[r_]["d_main"], dim_)
for nm_, (M_, dim_) in PROB.items():
    pr_ = el.Problem(M_)  # 입력 검증(대칭, 대각 0, 서로 다른 쌍은 양의 유한값)
    n_ = M_.shape[0]
    B_ = -0.5 * (np.eye(n_) - 1.0 / n_) @ (M_ ** 2) @ (np.eye(n_) - 1.0 / n_)
    w_ = np.sort(np.linalg.eigvalsh((B_ + B_.T) / 2))[::-1]
    assert (w_[:dim_] > 0).all(), f"{nm_}: 고전 MDS 상위 {dim_}개 고유값이 양수가 아니다"
print(len(PROB), "개 임베딩 입력:", list(PROB))
t0 = time.perf_counter()
RUNNER = el.PoolRunner(N_WORKERS, cache_dir=DIAG / "restart_cache")
try:
    EMB, CFGS = dl.embed_dims(PROB, RUNNER, CFG_RUN)
finally:
    RUNNER.close()
EMB_SECONDS = time.perf_counter() - t0
el.save_embeddings(EMB, DIAG / "emb", labels=None, cfg=CFG_RUN, D_by_name={nm_: M_ for nm_, (M_, _) in PROB.items()})
EMB_TABLE = dl.embedding_rows(EMB, CFGS)
csv_out(EMB_TABLE, "table_CD_embedding_all.csv")
print(f"임베딩 {len(EMB)}개 입력, 실행 {EMB_SECONDS / 60:.1f} 분 (재시작 합 {EMB_TABLE['초(재시작 합)'].sum() / 3600:.2f} 시간)")
display(EMB_TABLE[["입력", "dim", "n", "기록", "왜곡 비", "평균 상대 왜곡 쌍곡", "평균 상대 왜곡 유클리드", "stress 비(판정 재시작)", "수렴 쌍곡/유클리드", "상한에 닿은 점 수", "최대 반지름", "시도 수"]].round(4))
lap("C·D2 임베딩")

12 개 임베딩 입력: ['d_time_247_D5', 'd_time_247_D10', 'd_flow_main_D5', 'd_flow_main_D10', 'ctrl_main_0_D5', 'ctrl_main_1_D5', 'mig_obs_D2', 'mig_ctrl_main_0_D2', 'mig_ctrl_main_1_D2', 'mig_obs_D5', 'mig_ctrl_main_0_D5', 'mig_ctrl_main_1_D5']


임베딩 12개 입력, 실행 0.1 분 (재시작 합 5.40 시간)


,입력,dim,n,기록,왜곡 비,평균 상대 왜곡 쌍곡,평균 상대 왜곡 유클리드,stress 비(판정 재시작),수렴 쌍곡/유클리드,상한에 닿은 점 수,최대 반지름,시도 수
0,d_time_247_D5,5,247,쌍곡 이점 없음,0.9958,0.0300,0.0301,0.9902,"5/5, 5/5",0.0,1.1292,1
1,d_time_247_D10,10,247,쌍곡 이점 없음(평탄 극한),NaN,NaN,0.0235,NaN,"0/5, 5/5",NaN,NaN,1
2,d_flow_main_D5,5,247,쌍곡 이점 없음,0.9944,0.0682,0.0685,0.9895,"5/5, 5/5",0.0,0.2260,1
3,d_flow_main_D10,10,247,쌍곡 이점 없음(평탄 극한),1.0000,0.0544,0.0544,1.0000,"5/5, 5/5",0.0,0.0020,1
4,ctrl_main_0_D5,5,247,쌍곡 이점 없음(평탄 극한),1.0001,0.0485,0.0485,1.0000,"3/5, 5/5",0.0,0.0019,1
5,ctrl_main_1_D5,5,247,쌍곡 이점 없음,1.0000,0.0490,0.0490,0.9997,"3/5, 5/5",0.0,0.0073,1
6,mig_obs_D2,2,229,쌍곡 이점 있음,0.4911,0.1291,0.2629,0.3094,"5/5, 5/5",80.0,5.0000,1
7,mig_ctrl_main_0_D2,2,229,쌍곡 이점 있음,0.5880,0.1310,0.2228,0.3878,"5/5, 5/5",59.0,5.0000,1
8,mig_ctrl_main_1_D2,2,229,쌍곡 이점 있음,0.5813,0.1296,0.2229,0.3811,"5/5, 5/5",60.0,5.0000,1
9,mig_obs_D5,5,229,쌍곡 이점 있음,0.4493,0.0577,0.1285,0.2329,"5/5, 5/5",2.0,5.0000,1


[C·D2 임베딩] 4.1 s (누적 97 s)


### C 결과: 차원별 쌍곡·유클리드 왜곡 비

D = 2는 0S(`s4_real`) 결과를 그대로 쓴다. 왜곡 비 = 마무리(L-BFGS-B)가 수렴한 재시작 중 상대 stress가 가장 작은 재시작의 평균 상대 왜곡의 비(쌍곡 / 유클리드). 쌍곡 이점의 0S 기준은 비 ≤ 0.8이고 `d_flow`는 대조군 비보다도 작아야 한다. 쌍곡 재시작이 모두 마무리 반복 상한(maxiter 20,000)에 닿아 수렴한 것이 없으면 위 비를 낼 수 없으므로(D = 10의 `d_time`), 수렴 여부와 관계없이 상대 stress가 가장 작은 재시작으로 구한 보조값 `왜곡 비(수렴 무관)`을 함께 적는다. 이 보조값은 판정 규칙의 값이 아니다.

In [8]:
def row_from_emb(nm, base, dim):
    t = EMB_TABLE.set_index("입력").loc[nm]
    fb_ = dl.ratio_any_convergence(EMB[nm])  # 보조값: 마무리 수렴 여부와 관계없이 상대 stress가 가장 작은 재시작
    return {"입력": base, "dim": int(dim), "기록": t["기록"], "쌍곡 이점": t["쌍곡 이점"], "평균 상대 왜곡 쌍곡": t["평균 상대 왜곡 쌍곡"],
            "평균 상대 왜곡 유클리드": t["평균 상대 왜곡 유클리드"], "왜곡 비": t["왜곡 비"], "stress 비(판정 재시작)": t["stress 비(판정 재시작)"],
            "왜곡 비(수렴 무관)": fb_["ratio"], "평균 상대 왜곡 쌍곡(수렴 무관)": fb_["hyp_distortion"], "평균 상대 왜곡 유클리드(수렴 무관)": fb_["euc_distortion"],
            "stress 비(수렴 무관)": fb_["stress_ratio"], "수렴 무관 최소 stress 쌍곡 재시작의 마무리 수렴": fb_["hyp_converged"], "수렴 무관 최대 반지름": fb_["hyp_max_radius"],
            "수렴 쌍곡/유클리드": t["수렴 쌍곡/유클리드"], "최대 반지름": t["최대 반지름"], "상한에 닿은 점 수": t["상한에 닿은 점 수"], "시도 수": int(t["시도 수"]),
            "초(재시작 합)": float(t["초(재시작 합)"]), "출처": "04_diagnostics"}


C_BASE = ["d_time_247", "d_flow_main", "ctrl_main_0", "ctrl_main_1"]
C_ROWS = [dl.load_d2_summary(S4_REAL, b_) for b_ in C_BASE]
for b_ in ("d_time_247", "d_flow_main"):
    C_ROWS += [row_from_emb(f"{b_}_D{d_}", b_, d_) for d_ in (5, 10)]
C_ROWS += [row_from_emb(f"ctrl_main_{r_}_D5", f"ctrl_main_{r_}", 5) for r_ in (0, 1)]
C_TABLE = pd.DataFrame(C_ROWS)
C_TABLE["_o"] = C_TABLE["입력"].map({b_: i_ for i_, b_ in enumerate(C_BASE)})
C_TABLE = C_TABLE.sort_values(["_o", "dim"]).drop(columns="_o").reset_index(drop=True)
display(C_TABLE[["입력", "dim", "기록", "왜곡 비", "왜곡 비(수렴 무관)", "평균 상대 왜곡 쌍곡", "평균 상대 왜곡 유클리드", "stress 비(판정 재시작)", "수렴 쌍곡/유클리드", "최대 반지름", "상한에 닿은 점 수", "출처"]].round(4))
S0_ = {b_: json.loads((S4_REAL / f"summary_{b_}.json").read_text(encoding="utf-8")) for b_ in C_BASE}
ADV_C = {"D2": el.judge_flow_advantage(S0_["d_flow_main"], [S0_["ctrl_main_0"], S0_["ctrl_main_1"]], CFG),
         "D5": el.judge_flow_advantage(EMB["d_flow_main_D5"]["summary"], [EMB["ctrl_main_0_D5"]["summary"], EMB["ctrl_main_1_D5"]["summary"]], CFG)}
for k_, v_ in ADV_C.items():
    print(f"d_flow(주) 쌍곡 이점 점검 {k_}(대조군 2개): {v_['advantage']} | {v_['reason']}")
dl.plot_C(C_ROWS, OUT_FIG / "fig_C_dimensions.png")
csv_out(C_TABLE, "table_C_dimensions.csv")
R["C"] = {"label": dl.EXPLORATORY, "rows": {f"{r_['입력']}_D{r_['dim']}": r_ for r_ in C_ROWS}, "D2": {b_: S0_[b_]["ratio"] for b_ in C_BASE},
          "flow_advantage": ADV_C, "table": C_ROWS, "embedding_seconds": EMB_SECONDS, "restart_cpu_hours_all_C_D": float(EMB_TABLE["초(재시작 합)"].sum() / 3600),
          "protocol": "ADR 0013 4항 그대로(R_max 5, Adam 뒤 L-BFGS-B, 유클리드 5 + 쌍곡 5 재시작, 실패 시 쌍곡 U 재시작 시드 +5/+10). dim만 cfg['dim']으로 변경. D=2는 0S s4_real 결과"}
save_results()
lap("C 결과")

,입력,dim,기록,왜곡 비,왜곡 비(수렴 무관),평균 상대 왜곡 쌍곡,평균 상대 왜곡 유클리드,stress 비(판정 재시작),수렴 쌍곡/유클리드,최대 반지름,상한에 닿은 점 수,출처
0,d_time_247,2,쌍곡 이점 없음,0.9789,NaN,0.0717,0.0733,0.9742,NaN,2.2598,0.0,0S(s4_real)
1,d_time_247,5,쌍곡 이점 없음,0.9958,0.9958,0.0300,0.0301,0.9902,"5/5, 5/5",1.1292,0.0,04_diagnostics
2,d_time_247,10,쌍곡 이점 없음(평탄 극한),NaN,1.0000,NaN,0.0235,NaN,"0/5, 5/5",NaN,NaN,04_diagnostics
3,d_flow_main,2,쌍곡 이점 없음,0.9128,NaN,0.1461,0.1600,0.8747,NaN,5.0000,1.0,0S(s4_real)
4,d_flow_main,5,쌍곡 이점 없음,0.9944,0.9944,0.0682,0.0685,0.9895,"5/5, 5/5",0.2260,0.0,04_diagnostics
5,d_flow_main,10,쌍곡 이점 없음(평탄 극한),1.0000,1.0000,0.0544,0.0544,1.0000,"5/5, 5/5",0.0020,0.0,04_diagnostics
6,ctrl_main_0,2,쌍곡 이점 없음(평탄 극한),1.0002,NaN,0.0890,0.0890,1.0003,NaN,0.0016,0.0,0S(s4_real)
7,ctrl_main_0,5,쌍곡 이점 없음(평탄 극한),1.0001,1.0001,0.0485,0.0485,1.0000,"3/5, 5/5",0.0019,0.0,04_diagnostics
8,ctrl_main_1,2,쌍곡 이점 없음(평탄 극한),1.0000,NaN,0.0899,0.0899,1.0001,NaN,0.0005,0.0,0S(s4_real)
9,ctrl_main_1,5,쌍곡 이점 없음,1.0000,1.0000,0.0490,0.0490,0.9997,"3/5, 5/5",0.0073,0.0,04_diagnostics


d_flow(주) 쌍곡 이점 점검 D2(대조군 2개): 쌍곡 이점 없음 | 평균 상대 왜곡 비 0.9128 (문턱 0.8)
d_flow(주) 쌍곡 이점 점검 D5(대조군 2개): 쌍곡 이점 없음 | 평균 상대 왜곡 비 0.9944 (문턱 0.8)


[C 결과] 0.2 s (누적 97 s)


### D2 결과: 인구이동 `d_flow` 임베딩 (2차원, 5차원)

In [9]:
D_ROWS = [row_from_emb(f"{b_}_D{d_}", b_, d_) for d_ in (2, 5) for b_ in ("mig_obs", "mig_ctrl_main_0", "mig_ctrl_main_1")]
D_TABLE = pd.DataFrame(D_ROWS)
display(D_TABLE[["입력", "dim", "기록", "왜곡 비", "왜곡 비(수렴 무관)", "평균 상대 왜곡 쌍곡", "평균 상대 왜곡 유클리드", "stress 비(판정 재시작)", "수렴 쌍곡/유클리드", "최대 반지름", "상한에 닿은 점 수"]].round(4))
ADV_D = {}
for d_ in (2, 5):
    ADV_D[f"D{d_}"] = el.judge_flow_advantage(EMB[f"mig_obs_D{d_}"]["summary"], [EMB[f"mig_ctrl_main_{r_}_D{d_}"]["summary"] for r_ in (0, 1)], CFG)
    print(f"인구이동 d_flow(주) 쌍곡 이점 점검 D={d_}(대조군 2개): {ADV_D[f'D{d_}']['advantage']} | {ADV_D[f'D{d_}']['reason']}")
csv_out(D_TABLE, "table_D_embedding.csv")
R["D"]["rows"] = {f"{r_['입력']}_D{r_['dim']}": r_ for r_ in D_ROWS}
R["D"]["advantage"] = ADV_D
R["D"]["embedding_table"] = D_ROWS
save_results()
lap("D2 결과")

,입력,dim,기록,왜곡 비,왜곡 비(수렴 무관),평균 상대 왜곡 쌍곡,평균 상대 왜곡 유클리드,stress 비(판정 재시작),수렴 쌍곡/유클리드,최대 반지름,상한에 닿은 점 수
0,mig_obs,2,쌍곡 이점 있음,0.4911,0.4911,0.1291,0.2629,0.3094,"5/5, 5/5",5.0,80.0
1,mig_ctrl_main_0,2,쌍곡 이점 있음,0.5880,0.5880,0.1310,0.2228,0.3878,"5/5, 5/5",5.0,59.0
2,mig_ctrl_main_1,2,쌍곡 이점 있음,0.5813,0.5813,0.1296,0.2229,0.3811,"5/5, 5/5",5.0,60.0
3,mig_obs,5,쌍곡 이점 있음,0.4493,0.4493,0.0577,0.1285,0.2329,"5/5, 5/5",5.0,2.0
4,mig_ctrl_main_0,5,쌍곡 이점 있음,0.6594,0.6594,0.0800,0.1213,0.4266,"5/5, 5/5",5.0,5.0
5,mig_ctrl_main_1,5,쌍곡 이점 있음,0.6594,0.6594,0.0798,0.1210,0.4270,"5/5, 5/5",5.0,3.0


인구이동 d_flow(주) 쌍곡 이점 점검 D=2(대조군 2개): 쌍곡 이점 있음 | 비 0.4911 <= 0.8이고 대조군 비의 최솟값 0.5813보다 작다
인구이동 d_flow(주) 쌍곡 이점 점검 D=5(대조군 2개): 쌍곡 이점 있음 | 비 0.4493 <= 0.8이고 대조군 비의 최솟값 0.6594보다 작다
[D2 결과] 0.0 s (누적 97 s)


## E. 반경별 지역 수 증가 (공 성장) — 탐색적

B(r) = 노드별 반경 r 안 노드 수(자기 자신 포함)의 평균이다. 거리 10–90분위 구간에서 r을 같은 간격으로 100개 잡아 log B ~ log r(다항식형, 기울기 = 유효 차원)와 log B ~ r(지수형)의 R²를 비교한다. 이 구간에서 B의 양 끝값은 정의상 모든 입력에서 같으므로(쌍의 10%와 90%) 기울기는 q90/q10 거리비를 크게 반영하고, 형태는 R²와 곡률로 읽는다. 참고 기준선으로 같은 지역 대표점의 지리 직선거리(평면)를 같은 방법으로 적합한다.

In [10]:
org_ = pd.read_parquet(ORIGINS, columns=["unit_code", "x_m", "y_m"]).set_index("unit_code").loc[inp["units250"]["unit_code"].tolist()]
X250, Y250 = org_["x_m"].to_numpy(), org_["y_m"].to_numpy()
x249, y249 = dl.group_centroids(X250, Y250, inp["units250"]["idx249"].to_numpy(), inp["pop250"], 249)
i247_ = inp["units247"]["idx249"].to_numpy()
GEO247 = dl.geo_distance_km(x249[i247_], y249[i247_])
x229, y229 = dl.group_centroids(X250, Y250, MIG["groups250"], inp["pop250"], 229)
GEO229 = dl.geo_distance_km(x229, y229)
E_IN = {"d_time": inp["d_time_247"], "d_flow_commute": inp["d_flow_main_247"], "d_flow_commute_alt": inp["d_flow_alt_247"], "d_flow_migration": FD_MIG["d_main"],
        "d_time_229": MIG["d_time"], "geo_247": GEO247, "geo_229": GEO229, "A1_commute": FD_A1["d_main"], "A1_migration": FD_A1_MIG["d_main"]}
E = {k_: dl.growth_fit(D_) for k_, D_ in E_IN.items()}
E_Q = {k_: {kk_: vv_ for kk_, vv_ in dl.growth_fit(D_, grid="q").items() if kk_ not in ("r", "B")} for k_, D_ in E_IN.items()}


def avg_fit(Ms):
    fits = [dl.growth_fit(M_) for M_ in Ms]
    out = {"n": len(fits)}
    for kk_ in ("slope_pow", "r2_pow", "rate_exp_diam", "r2_exp", "r2_diff"):
        out[kk_] = float(np.mean([f_[kk_] for f_ in fits]))
        out[kk_ + "_sd"] = float(np.std([f_[kk_] for f_ in fits], ddof=1))
    out["prefer_counts"] = {p_: int(sum(f_["prefer"] == p_ for f_ in fits)) for p_ in ("다항식형", "지수형", "구분 불가")}
    return out


E["ctrl_commute_0S"] = avg_fit(inp["controls_d_main_247"])
E["ctrl_migration"] = avg_fit([c_["d_main"] for c_ in CTRL_MIG])
E_NAMES = {"d_time": "d_time 247 (OSM 통행시간, 분)", "d_flow_commute": "d_flow 주, 통근·통학 247", "d_flow_commute_alt": "d_flow 대안, 통근·통학 247",
           "d_flow_migration": "d_flow 주, 인구이동 229", "d_time_229": "d_time 229 (참고)", "geo_247": "지리 직선거리 247 (참고 기준선)", "geo_229": "지리 직선거리 229 (참고 기준선)",
           "A1_commute": "A1 같은 지지 집합 대조군 d_flow 주, 통근", "A1_migration": "A1형 대조군 d_flow 주, 인구이동"}
rows_ = []
for k_, nm_ in E_NAMES.items():
    g_ = E[k_]
    rows_.append({"입력": nm_, "n": g_["n"], "r 구간": f"{g_['r_lo']:.3g}–{g_['r_hi']:.3g}", "q90/q10": round(g_["r_hi"] / g_["r_lo"], 2), "유효 차원(기울기)": g_["slope_pow"], "R² 다항식형": g_["r2_pow"],
                  "성장률 x 지름": g_["rate_exp_diam"], "R² 지수형": g_["r2_exp"], "R² 차이": g_["r2_diff"], "선호": g_["prefer"], "적합 품질": g_["quality"],
                  "분위수 격자 R² 다항/지수": f"{E_Q[k_]['r2_pow']:.4f}/{E_Q[k_]['r2_exp']:.4f}"})
for k_, nm_ in (("ctrl_commute_0S", "0S 중력 대조군 20개 평균, 통근"), ("ctrl_migration", "중력 대조군 20개 평균, 인구이동")):
    g_ = E[k_]
    rows_.append({"입력": nm_, "n": g_["n"], "r 구간": "", "q90/q10": None, "유효 차원(기울기)": g_["slope_pow"], "R² 다항식형": g_["r2_pow"], "성장률 x 지름": g_["rate_exp_diam"],
                  "R² 지수형": g_["r2_exp"], "R² 차이": g_["r2_diff"], "선호": str(g_["prefer_counts"]), "적합 품질": "", "분위수 격자 R² 다항/지수": ""})
E_TABLE = pd.DataFrame(rows_)
display(E_TABLE.round(4))
dl.plot_E([("d_time 247개 (OSM 통행시간)", E["d_time"], dl.SERIES[0], E["geo_247"]), ("d_flow 주, 통근·통학 247개", E["d_flow_commute"], dl.SERIES[1], E["geo_247"]),
           ("d_flow 주, 인구이동 229개", E["d_flow_migration"], dl.SERIES[2], E["geo_229"])], OUT_FIG / "fig_E_ball_growth.png")
csv_out(E_TABLE, "table_E_growth.csv")
R["E"] = {"label": dl.EXPLORATORY, **E, "q_grid": E_Q, "names": E_NAMES,
          "method": "B(r) = 노드별 반경 r 안 노드 수(자기 자신 포함)의 평균. r = 거리 10–90분위를 같은 간격 100개(격자 q는 분위수 같은 간격). OLS. 선호: R² 차이 0.002 초과인 쪽, 아니면 구분 불가"}
save_results()
lap("E 공 성장")

,입력,n,r 구간,q90/q10,유효 차원(기울기),R² 다항식형,성장률 x 지름,R² 지수형,R² 차이,선호,적합 품질,분위수 격자 R² 다항/지수
0,"d_time 247 (OSM 통행시간, 분)",247,48.5–248,5.12,1.3681,0.9989,5.2123,0.9496,0.0494,다항식형,양호,0.9988/0.9482
1,"d_flow 주, 통근·통학 247",247,5.57–18.5,3.32,1.6709,0.9951,3.9093,0.9587,0.0365,다항식형,양호,0.9957/0.9610
2,"d_flow 대안, 통근·통학 247",247,5.45–18.2,3.35,1.6596,0.9944,3.9004,0.9556,0.0388,다항식형,양호,0.9950/0.9583
3,"d_flow 주, 인구이동 229",229,5.03–8.96,1.78,3.8992,0.9521,6.6225,0.9160,0.0361,다항식형,불량(둘 다 R² < 0.98),0.9543/0.9171
4,d_time 229 (참고),229,53.7–254,4.74,1.4337,0.9982,6.3666,0.9480,0.0502,다항식형,양호,0.9980/0.9462
5,지리 직선거리 247 (참고 기준선),247,43.5–307,7.06,1.1469,0.9987,4.0608,0.9336,0.0651,다항식형,양호,0.9987/0.9315
6,지리 직선거리 229 (참고 기준선),229,48.7–315,6.46,1.1955,0.9981,4.5658,0.9317,0.0664,다항식형,양호,0.9981/0.9295
7,"A1 같은 지지 집합 대조군 d_flow 주, 통근",247,5.41–25.3,4.68,1.3550,0.9921,3.2935,0.9228,0.0693,다항식형,양호,0.9922/0.9201
8,"A1형 대조군 d_flow 주, 인구이동",229,4.61–9.44,2.05,2.9322,0.9544,5.3539,0.9083,0.0461,다항식형,불량(둘 다 R² < 0.98),0.9564/0.9102
9,"0S 중력 대조군 20개 평균, 통근",20,,NaN,1.3680,0.9930,3.3229,0.9260,0.0670,"{'다항식형': 20, '지수형': 0, '구분 불가': 0}",,


[E 공 성장] 1.1 s (누적 98 s)


## F. 가설별 지지 여부 — 탐색적

규칙은 `diag_lib.hypothesis_verdicts`에 적었고 각 행의 "규칙" 열에도 있다. 사전에 고정한 규칙이 아니며 0S 판정을 바꾸지 않는다.

In [11]:
VERD = dl.hypothesis_verdicts(R)
VERD_TABLE = pd.DataFrame(VERD)
display(VERD_TABLE[["no", "가설", "판정", "근거"]])
for v_ in VERD:
    print(f"({v_['no']}) {v_['가설']} -> {v_['판정']}\n    근거: {v_['근거']}\n    규칙: {v_['규칙']}")
csv_out(VERD_TABLE, "table_F_verdicts.csv")
R["F"] = VERD

git_dirty = subprocess.run(["git", "status", "--porcelain", "--", "."], capture_output=True, text=True, cwd=NB_DIR).stdout.strip() != ""
R["meta"].update({
    "created": datetime.datetime.now().isoformat(timespec="seconds"), "smoke": SMOKE, "n_workers": N_WORKERS,
    "env": {"python": sys.version.split()[0], "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__, "torch": torch.__version__, "geoopt": geoopt.__version__, "numba": el.numba.__version__},
    "config_hash": el.config_hash(CFG), "config": CFG,
    "data_snapshot": {"s4_inputs_md5": inp["md5"], "mig2025_xlsx_md5": MIG_MD5, "origins_parquet_md5": dl.md5_file(ORIGINS)},
    "code_version": {"git_head": dl.git_head(NB_DIR), "working_tree_dirty": git_dirty, "embed_lib.py": dl.md5_file(NB_DIR / "embed_lib.py"), "diag_lib.py": dl.md5_file(NB_DIR / "diag_lib.py")},
    "seed": {
        "embedding_restarts": "0S와 같다: 유클리드 고전 MDS 1 + U(+-0.1) 시드 0..3, 쌍곡 줄인 초기값 2 + U 시드 0..2, 실패 시 쌍곡 U 시드 +5, +10 (D차원 U = default_rng(seed).uniform(-0.1, 0.1, (n, D)))",
        "A2_resample": f"np.random.default_rng([{el.RESAMPLE_STREAM}, r]) r = 0..19(연결된 앞 20개), 표본 배율 k = {MATCH['k']:.4f}",
        "A3_sweep": "같은 난수열 r = 0..5", "D_controls": f"같은 난수열 r = 0..19, 표본 비율 1.0", "bisection_seeds": "r = 0..19의 평균 엣지 수(결정적)"},
    "mds_max_norm_note": "차원을 바꿔도 mds_max_norm(0.1*sqrt(2))과 m1, m2는 0S 값 그대로 둔다(2차원 규약을 변경하지 않음)",
})
save_results()
lap("F 요약·저장")
print("전체 실행 시간:", round(time.time() - T_START), "초 | 결과:", RESULT_JSON)

,no,가설,판정,근거
0,1,대조군 밀도 차이가 위계 판정을 좌우했다,기각,"δ̄(주, 전체): 관측 0.301, 0S 대조군 0.263(엣지 5217), A1 0.260(엣지 6815), A2 0.259(엣지 6781, 표본 배율 x2041). 관측-대조군 간격 0.039 -> A1 0.041, A2 0.042(축소 -8%). 위계 판정 뒤집힘: A1 False, A2 False. 대조군 엣지 5209 -> 6782에서 δ̄ 0.262 -> 0.259"
1,2,통근이 국지적이라 격자형 거리다,지지,"통근 d_flow 공 성장: 다항식형 R² 0.995(유효 차원 1.67) 대 지수형 R² 0.959 -> 다항식형. 0S 중력 대조군 20개 평균은 R² 0.993/0.926, 차원 1.37(같은 지지 집합 A1 1.36); 지리 직선거리 기준선 차원 1.15. 통행량 가중 평균 통행시간 21.5분. δ̄(주, 전체)는 관측 0.301, 0S 중력 대조군 0.263로 관측이 더 크다(덜 트리형)"
2,3,2차원이 너무 좁다,기각,"왜곡 비(쌍곡/유클리드) D=2 -> 5 -> 10: d_time_247 0.979 -> 0.996 -> 1.000*, d_flow_main 0.913 -> 0.994 -> 1.000. 대조군 D=5: 1.000, 1.000. (*는 수렴한 쌍곡 재시작이 없어 수렴 무관 최소 stress 재시작으로 구한 보조값)"
3,4,전국 규모(인구이동)에서는 위계나 쌍곡성이 보인다,부분 지지,"인구이동 δ̄(주, 전체) 0.384 대 대조군 20개 0.370–0.372(조건 통과 0/4, 위계 False); 같은 지지 집합 대조군 0.334. 왜곡 비 D=2 0.491(대조군 0.588, 0.581), D=5 0.449(대조군 0.659, 0.659). R_max 상한에 닿은 점(D=2, 229개 중) 관측 80, 대조군 59, 60개. d_time 229 δ̄ 0.249"
4,5,d_time은 평면형 성장이다,지지,"d_time 공 성장: 다항식형 R² 0.999(유효 차원 1.37) 대 지수형 R² 0.950 -> 다항식형. 지리 직선거리 기준선: R² 0.999/0.934, 차원 1.15. 10–90분위 구간의 기울기는 q90/q10 거리비(5.12)로 정해지는 면이 커서 2보다 작다(국토가 길쭉하고 경계 효과가 있다)"


(1) 대조군 밀도 차이가 위계 판정을 좌우했다 -> 기각
    근거: δ̄(주, 전체): 관측 0.301, 0S 대조군 0.263(엣지 5217), A1 0.260(엣지 6815), A2 0.259(엣지 6781, 표본 배율 x2041). 관측-대조군 간격 0.039 -> A1 0.041, A2 0.042(축소 -8%). 위계 판정 뒤집힘: A1 False, A2 False. 대조군 엣지 5209 -> 6782에서 δ̄ 0.262 -> 0.259
    규칙: A1·A2 모두에서 4개 조건 위계 통과(판정 뒤집힘)면 지지, 뒤집히지 않고 관측-대조군 간격이 0S의 절반 이하로 줄면 부분 지지, 그 밖은 기각
(2) 통근이 국지적이라 격자형 거리다 -> 지지
    근거: 통근 d_flow 공 성장: 다항식형 R² 0.995(유효 차원 1.67) 대 지수형 R² 0.959 -> 다항식형. 0S 중력 대조군 20개 평균은 R² 0.993/0.926, 차원 1.37(같은 지지 집합 A1 1.36); 지리 직선거리 기준선 차원 1.15. 통행량 가중 평균 통행시간 21.5분. δ̄(주, 전체)는 관측 0.301, 0S 중력 대조군 0.263로 관측이 더 크다(덜 트리형)
    규칙: 통근 d_flow의 다항식형 R²가 지수형보다 0.002 넘게 크고 유효 차원이 1.5–3이면 지지, 지수형이 더 맞으면 기각, 그 밖은 부분 지지
(3) 2차원이 너무 좁다 -> 기각
    근거: 왜곡 비(쌍곡/유클리드) D=2 -> 5 -> 10: d_time_247 0.979 -> 0.996 -> 1.000*, d_flow_main 0.913 -> 0.994 -> 1.000. 대조군 D=5: 1.000, 1.000. (*는 수렴한 쌍곡 재시작이 없어 수렴 무관 최소 stress 재시작으로 구한 보조값)
    규칙: D = 5 또는 10에서 d_time 또는 d_flow의 왜곡 비가 0.8 이하면 지지, 비가 2차원보다 0.05 넘게 내려가면 부분 지지, 그 밖은 기각
(4)